## Chroma DB
chroma DB is an open-source, AI-native vector database designed specifically to store and query vector embeddings.
Unlike FAISS, which is a low-level mathematical engine (a library), Chroma is a complete database wrapper. It includes a developer-friendly API, built-in embedding models, and metadata filtering right out of the box. It is highly popular for prototyping, local LLM applications, and Retrieval-Augmented Generation (RAG) because it can be spun up with just a few lines of code without configuring external servers.


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [3]:
#Load the text or infromation

loader=TextLoader("speech.txt")

docs=loader.load()
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [4]:
#Chunk tthe document for Model effeciency
chunk_document=CharacterTextSplitter( chunk_size=200,chunk_overlap=20)

chunk_document

In [5]:
final_document=chunk_document.split_documents(docs)

Created a chunk of size 470, which is longer than the specified 200
Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 668, which is longer than the specified 200
Created a chunk of size 982, which is longer than the specified 200
Created a chunk of size 789, which is longer than the specified 200


In [6]:
##Embedding the data and store it into the FAISS datastore

embedding=OllamaEmbeddings(model="nomic-embed-text")
embedding

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [14]:
vectore_store_db=Chroma.from_documents(documents=final_document,embedding=embedding)
vectore_store_db

## Query
infromation from the Dataset

In [16]:
query="What does the speaker believe is the main reason the United States should enter the war?"
docs=vectore_store_db.similarity_search(query)

In [17]:
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## Save its into the local disk

In [19]:
vectore_store_db=Chroma.from_documents(documents=final_document,embedding=embedding,persist_directory="./chroma_db")

## Load from the local disk

In [25]:
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs=db2.similarity_search(query)

In [26]:
docs

[Document(id='30bb75cd-46f4-4b18-a654-b487ac23ec72', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='72043f10-26b9-46ad-b946-fda9ba8eaaee', metadata={'source': 'speech.txt'}, page_content='To 

In [29]:
## Simnlarity_seaech_with-score

docs=db2.similarity_search_with_score(query)

In [30]:
docs

[(Document(id='30bb75cd-46f4-4b18-a654-b487ac23ec72', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  0.8565471172332764),
 (Document(id='72043f10-26b9-46ad-b946-fda9ba8eaaee', metadata={'source': 'speech

In [31]:
### Retriever option
retriever=vectore_store_db.as_retriever()
retriever.invoke(query)[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'